## Quick check if I'm connected to the colab GPU kernel (T4)

In [1]:
import os
import sys

print("python_executable:", sys.executable)
print("google.colab in modules:", "google.colab" in sys.modules)
print("COLAB_GPU env:", os.environ.get("COLAB_GPU"))

# Estimate system RAM without extra dependencies
try:
    page_size = os.sysconf("SC_PAGE_SIZE")
    phys_pages = os.sysconf("SC_PHYS_PAGES")
    total_gb = (page_size * phys_pages) / (1024 ** 3)
    print(f"system_ram_gb: {total_gb:.2f}")
except Exception as e:
    print("ram_check_error:", str(e))

try:
    import torch
    print("torch_version:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_check_error:", str(e))

# Lightweight Aya availability check (config + tokenizer only, no 8B weights)
try:
    from transformers import AutoConfig, AutoTokenizer
    cfg = AutoConfig.from_pretrained("CohereForAI/aya-expanse-8b")
    tok = AutoTokenizer.from_pretrained("CohereForAI/aya-expanse-8b")
    print("aya_config_loaded:", True)
    print("aya_model_type:", getattr(cfg, "model_type", "unknown"))
    print("aya_tokenizer_vocab_size:", len(tok))
except Exception as e:
    print("aya_lightweight_check_error:", str(e))

print("estimated_fp32_memory_for_8b_params_gb: ~32+")
print("estimated_fp16_memory_for_8b_params_gb: ~16+ (plus overhead)")

python_executable: /usr/bin/python3
google.colab in modules: True
COLAB_GPU env: 1
system_ram_gb: 12.67
torch_version: 2.10.0+cu128
cuda_available: True
gpu_name: Tesla T4
aya_lightweight_check_error: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/CohereForAI/aya-expanse-8b.
401 Client Error. (Request ID: Root=1-69da3288-756e05e42bea10c5267fb945;ed09c673-8bfd-4cd5-b5cd-f47aa9921e9d)

Cannot access gated repo for url https://huggingface.co/CohereForAI/aya-expanse-8b/resolve/main/config.json.
Access to model CohereLabs/aya-expanse-8b is restricted. You must have access to it and be authenticated to access it. Please log in.
estimated_fp32_memory_for_8b_params_gb: ~32+
estimated_fp16_memory_for_8b_params_gb: ~16+ (plus overhead)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [2]:
# Run this cell anytime to check if your code is using the GPU
!nvidia-smi

import torch
if torch.cuda.is_available():
    print(f"\nMemory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Sat Apr 11 10:29:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Probing Experiment: Semantic Role Encoding in LLMs

This notebook implements a probing experiment to test whether the `aya-expanse-8b` language model
encodes semantic roles (Agent vs Patient) in its hidden representations.

**Approach:**
1. Extract hidden states for target nouns in sentences with known Agent/Patient labels.
2. Train a linear classifier (logistic regression) on those hidden states.
3. Evaluate with cross-validation and on a held-out test set.

**Design choices to avoid trivial heuristics:**
- Same nouns appear as both Agent and Patient across examples.
- Both active constructions (Agent-first) and passive constructions (Patient-first) are included.
- Word position or grammatical role (subject, object) is therefore not a reliable cue for semantic role.

**Requirements:** GPU needed: run with Google Colab kernel (MyColabGPU, T4).

## 1. Install Dependencies and get Access to Aya model

In [3]:
# Install required packages (uncomment if running in a fresh Colab environment)
%pip install transformers torch scikit-learn accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.2 MB/s eta 0:00:00:00:0100:01


In [2]:
# Hugging Face authentication for gated Aya model access
# Run this once per new kernel session.
from getpass import getpass
from huggingface_hub import login
from transformers import AutoConfig

token = getpass("Paste your Hugging Face read token, then press Enter: ").strip()
if not token:
    raise ValueError("No token entered. Please rerun this cell and paste your token.")
if not token.startswith("hf_"):
    raise ValueError("Token format looks invalid. It should usually start with 'hf_'.")

login(token=token, add_to_git_credential=False)
# Quick access check for the gated model
cfg = AutoConfig.from_pretrained("CohereForAI/aya-expanse-8b")
print("Hugging Face auth OK. Model type:", cfg.model_type)

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

Hugging Face auth OK. Model type: cohere


## 2. Imports

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from typing import List, Tuple, Dict, Optional

## 3. Model Loading

In [4]:
def load_model_and_tokenizer(
    model_name: str = "CohereForAI/aya-expanse-8b",
    device: Optional[str] = None,
) -> Tuple[AutoModelForCausalLM, AutoTokenizer, str]:
    """
    Load the language model and its tokenizer with output_hidden_states enabled.

    Args:
        model_name: HuggingFace model identifier.
        device: 'cuda', 'cpu', or None (auto-detect).

    Returns:
        Tuple of (model, tokenizer, device_string).
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading tokenizer for '{model_name}'...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print(f"Loading model on device: {device}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        output_hidden_states=True,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    model.eval()

    print("Model loaded successfully.")
    return model, tokenizer, device

## 4. Hidden State Extraction

In [5]:
def find_token_positions(
    tokenizer: AutoTokenizer,
    input_ids: torch.Tensor,
    target_word: str,
) -> List[int]:
    """
    Find the token positions (indices) for a target word in the tokenized input.

    Handles sub-word tokenization by searching for all token spans that
    correspond to the target word.

    Args:
        tokenizer: The tokenizer used for encoding.
        input_ids: 1-D tensor of token IDs (no batch dimension).
        target_word: The surface form of the word to locate.

    Returns:
        List of token position indices that correspond to the target word.
        Returns an empty list if the word is not found.
    """
    # Encode the target word with and without a leading space to capture
    # both sentence-initial and sentence-internal occurrences.
    tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist())

    # Build a mapping from position to the decoded string for that token,
    # then slide a window to find where the target word starts.
    target_lower = target_word.lower()

    # Reconstruct the decoded text for each position, stripping common
    # BPE prefix characters (▁, Ġ, ##) for comparison.
    clean_tokens = [
        t.lstrip("▁Ġ#").lower() if t is not None else "" for t in tokens
    ]

    positions = []
    for i, ct in enumerate(clean_tokens):
        if ct == target_lower:
            positions.append(i)

    # If not found via exact cleaned-token match, fall back to a
    # sliding-window sub-word merge approach.
    if not positions:
        for i in range(len(tokens)):
            merged = ""
            span = []
            for j in range(i, len(tokens)):
                merged += clean_tokens[j]
                span.append(j)
                if merged == target_lower:
                    positions = span
                    break
                if len(merged) >= len(target_lower):
                    break
            if positions:
                break

    return positions


def extract_hidden_state(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    sentence: str,
    target_word: str,
    layer: int = -1,
    device: str = "cpu",
) -> np.ndarray:
    """
    Extract the hidden-state vector for a target word in a sentence.

    When the target word is split into multiple sub-word tokens, the
    representations are averaged across those tokens.

    Args:
        model: The language model (must have output_hidden_states=True).
        tokenizer: Corresponding tokenizer.
        sentence: Full input sentence string.
        target_word: The word whose hidden state we want to extract.
        layer: Which transformer layer to use (-1 = last layer).
        device: Device string ('cuda' or 'cpu').

    Returns:
        1-D numpy array of shape (hidden_size,).

    Raises:
        ValueError: If the target word cannot be located in the tokenized input.
    """
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs["input_ids"][0]  # shape: (seq_len,)

    positions = find_token_positions(tokenizer, input_ids, target_word)
    if not positions:
        raise ValueError(
            f"Target word '{target_word}' not found in tokenized sentence: "
            f"'{sentence}'\nTokens: {tokenizer.convert_ids_to_tokens(input_ids.tolist())}"
        )

    # Move inputs to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # hidden_states is a tuple of (num_layers + 1) tensors,
    # each of shape (batch, seq_len, hidden_size).
    hidden_states = outputs.hidden_states  # tuple length = n_layers + 1
    layer_states = hidden_states[layer]  # shape: (1, seq_len, hidden_size)

    # Extract and average across target token positions
    token_vecs = layer_states[0, positions, :]  # shape: (n_tokens, hidden_size)
    representation = token_vecs.mean(dim=0).float().cpu().numpy()  # (hidden_size,)

    return representation

## 5. Dataset Construction

Each example is a tuple: `(sentence, target_word, label)` where label is `"Agent"` or `"Patient"`.

Key design principles:
- **Same nouns** (e.g., *dog*, *cat*) appear as **both Agent and Patient** across different sentences.
- **Active sentences**: the Agent appears in subject (first) position.
- **Passive sentences**: the Patient appears in subject (first) position.
- This prevents the classifier from relying on word identity or surface position.

In [6]:
def create_dataset() -> List[Tuple[str, str, str]]:
    """
    Create a dataset of (sentence, target_word, label) triples.

    The dataset contains 152 examples:
    - Active constructions  (Agent in subject position)
    - Passive constructions (Patient in subject position)

    Every noun appears as both Agent and Patient to prevent the model
    from exploiting word-identity or positional heuristics.

    Returns:
        List of (sentence, target_word, label) tuples.
        label is either 'Agent' or 'Patient'.
    """
    # ------------------------------------------------------------------ #
    # Raw sentence templates: (sentence, target_word, label)              #
    # Active:  <Agent> <verb> <Patient>                                   #
    # Passive: <Patient> was <past-participle> by <Agent>                 #
    # ------------------------------------------------------------------ #

    examples = [
        # ---- dog / cat ------------------------------------------------
        ("The dog chased the cat.", "dog", "Agent"),
        ("The dog chased the cat.", "cat", "Patient"),
        ("The cat was chased by the dog.", "cat", "Patient"),
        ("The cat was chased by the dog.", "dog", "Agent"),
        ("The cat scratched the dog.", "cat", "Agent"),
        ("The cat scratched the dog.", "dog", "Patient"),
        ("The dog was scratched by the cat.", "dog", "Patient"),
        ("The dog was scratched by the cat.", "cat", "Agent"),
        # ---- man / woman ---------------------------------------------
        ("The man saw the woman.", "man", "Agent"),
        ("The man saw the woman.", "woman", "Patient"),
        ("The woman was seen by the man.", "woman", "Patient"),
        ("The woman was seen by the man.", "man", "Agent"),
        ("The woman helped the man.", "woman", "Agent"),
        ("The woman helped the man.", "man", "Patient"),
        ("The man was helped by the woman.", "man", "Patient"),
        ("The man was helped by the woman.", "woman", "Agent"),
        # ---- child / teacher -----------------------------------------
        ("The child found the teacher.", "child", "Agent"),
        ("The child found the teacher.", "teacher", "Patient"),
        ("The teacher was found by the child.", "teacher", "Patient"),
        ("The teacher was found by the child.", "child", "Agent"),
        ("The teacher scolded the child.", "teacher", "Agent"),
        ("The teacher scolded the child.", "child", "Patient"),
        ("The child was scolded by the teacher.", "child", "Patient"),
        ("The child was scolded by the teacher.", "teacher", "Agent"),
        # ---- bird / fox -----------------------------------------------
        ("The bird spotted the fox.", "bird", "Agent"),
        ("The bird spotted the fox.", "fox", "Patient"),
        ("The fox was spotted by the bird.", "fox", "Patient"),
        ("The fox was spotted by the bird.", "bird", "Agent"),
        ("The fox chased the bird.", "fox", "Agent"),
        ("The fox chased the bird.", "bird", "Patient"),
        ("The bird was chased by the fox.", "bird", "Patient"),
        ("The bird was chased by the fox.", "fox", "Agent"),
        # ---- student / professor -------------------------------------
        ("The student greeted the professor.", "student", "Agent"),
        ("The student greeted the professor.", "professor", "Patient"),
        ("The professor was greeted by the student.", "professor", "Patient"),
        ("The professor was greeted by the student.", "student", "Agent"),
        ("The professor praised the student.", "professor", "Agent"),
        ("The professor praised the student.", "student", "Patient"),
        ("The student was praised by the professor.", "student", "Patient"),
        ("The student was praised by the professor.", "professor", "Agent"),
        # ---- lion / zebra --------------------------------------------
        ("The lion caught the zebra.", "lion", "Agent"),
        ("The lion caught the zebra.", "zebra", "Patient"),
        ("The zebra was caught by the lion.", "zebra", "Patient"),
        ("The zebra was caught by the lion.", "lion", "Agent"),
        ("The zebra kicked the lion.", "zebra", "Agent"),
        ("The zebra kicked the lion.", "lion", "Patient"),
        ("The lion was kicked by the zebra.", "lion", "Patient"),
        ("The lion was kicked by the zebra.", "zebra", "Agent"),
        # ---- doctor / patient (role noun used intentionally) ---------
        ("The doctor examined the nurse.", "doctor", "Agent"),
        ("The doctor examined the nurse.", "nurse", "Patient"),
        ("The nurse was examined by the doctor.", "nurse", "Patient"),
        ("The nurse was examined by the doctor.", "doctor", "Agent"),
        ("The nurse assisted the doctor.", "nurse", "Agent"),
        ("The nurse assisted the doctor.", "doctor", "Patient"),
        ("The doctor was assisted by the nurse.", "doctor", "Patient"),
        ("The doctor was assisted by the nurse.", "nurse", "Agent"),
        # ---- cat / mouse ---------------------------------------------
        ("The cat caught the mouse.", "cat", "Agent"),
        ("The cat caught the mouse.", "mouse", "Patient"),
        ("The mouse was caught by the cat.", "mouse", "Patient"),
        ("The mouse was caught by the cat.", "cat", "Agent"),
        ("The mouse bit the cat.", "mouse", "Agent"),
        ("The mouse bit the cat.", "cat", "Patient"),
        ("The cat was bitten by the mouse.", "cat", "Patient"),
        ("The cat was bitten by the mouse.", "mouse", "Agent"),
        # ---- boy / girl -----------------------------------------------
        ("The boy pushed the girl.", "boy", "Agent"),
        ("The boy pushed the girl.", "girl", "Patient"),
        ("The girl was pushed by the boy.", "girl", "Patient"),
        ("The girl was pushed by the boy.", "boy", "Agent"),
        ("The girl hugged the boy.", "girl", "Agent"),
        ("The girl hugged the boy.", "boy", "Patient"),
        ("The boy was hugged by the girl.", "boy", "Patient"),
        ("The boy was hugged by the girl.", "girl", "Agent"),
        # ---- wolf / deer ---------------------------------------------
        ("The wolf frightened the deer.", "wolf", "Agent"),
        ("The wolf frightened the deer.", "deer", "Patient"),
        ("The deer was frightened by the wolf.", "deer", "Patient"),
        ("The deer was frightened by the wolf.", "wolf", "Agent"),
        ("The deer kicked the wolf.", "deer", "Agent"),
        ("The deer kicked the wolf.", "wolf", "Patient"),
        ("The wolf was kicked by the deer.", "wolf", "Patient"),
        ("The wolf was kicked by the deer.", "deer", "Agent"),
        # ---- chef / guest --------------------------------------------
        ("The chef served the guest.", "chef", "Agent"),
        ("The chef served the guest.", "guest", "Patient"),
        ("The guest was served by the chef.", "guest", "Patient"),
        ("The guest was served by the chef.", "chef", "Agent"),
        ("The guest praised the chef.", "guest", "Agent"),
        ("The guest praised the chef.", "chef", "Patient"),
        ("The chef was praised by the guest.", "chef", "Patient"),
        ("The chef was praised by the guest.", "guest", "Agent"),
        # ---- officer / suspect ---------------------------------------
        ("The officer arrested the suspect.", "officer", "Agent"),
        ("The officer arrested the suspect.", "suspect", "Patient"),
        ("The suspect was arrested by the officer.", "suspect", "Patient"),
        ("The suspect was arrested by the officer.", "officer", "Agent"),
        ("The suspect challenged the officer.", "suspect", "Agent"),
        ("The suspect challenged the officer.", "officer", "Patient"),
        ("The officer was challenged by the suspect.", "officer", "Patient"),
        ("The officer was challenged by the suspect.", "suspect", "Agent"),
        # ---- horse / rider -------------------------------------------
        ("The horse threw the rider.", "horse", "Agent"),
        ("The horse threw the rider.", "rider", "Patient"),
        ("The rider was thrown by the horse.", "rider", "Patient"),
        ("The rider was thrown by the horse.", "horse", "Agent"),
        ("The rider guided the horse.", "rider", "Agent"),
        ("The rider guided the horse.", "horse", "Patient"),
        ("The horse was guided by the rider.", "horse", "Patient"),
        ("The horse was guided by the rider.", "rider", "Agent"),
        # ---- scientist / assistant -----------------------------------
        ("The scientist observed the assistant.", "scientist", "Agent"),
        ("The scientist observed the assistant.", "assistant", "Patient"),
        ("The assistant was observed by the scientist.", "assistant", "Patient"),
        ("The assistant was observed by the scientist.", "scientist", "Agent"),
        ("The assistant supported the scientist.", "assistant", "Agent"),
        ("The assistant supported the scientist.", "scientist", "Patient"),
        ("The scientist was supported by the assistant.", "scientist", "Patient"),
        ("The scientist was supported by the assistant.", "assistant", "Agent"),
        # ---- eagle / rabbit ------------------------------------------
        ("The eagle grabbed the rabbit.", "eagle", "Agent"),
        ("The eagle grabbed the rabbit.", "rabbit", "Patient"),
        ("The rabbit was grabbed by the eagle.", "rabbit", "Patient"),
        ("The rabbit was grabbed by the eagle.", "eagle", "Agent"),
        ("The rabbit evaded the eagle.", "rabbit", "Agent"),
        ("The rabbit evaded the eagle.", "eagle", "Patient"),
        ("The eagle was evaded by the rabbit.", "eagle", "Patient"),
        ("The eagle was evaded by the rabbit.", "rabbit", "Agent"),
        # ---- pilot / passenger ---------------------------------------
        ("The pilot warned the passenger.", "pilot", "Agent"),
        ("The pilot warned the passenger.", "passenger", "Patient"),
        ("The passenger was warned by the pilot.", "passenger", "Patient"),
        ("The passenger was warned by the pilot.", "pilot", "Agent"),
        ("The passenger thanked the pilot.", "passenger", "Agent"),
        ("The passenger thanked the pilot.", "pilot", "Patient"),
        ("The pilot was thanked by the passenger.", "pilot", "Patient"),
        ("The pilot was thanked by the passenger.", "passenger", "Agent"),
        # ---- shark / diver -------------------------------------------
        ("The shark frightened the diver.", "shark", "Agent"),
        ("The shark frightened the diver.", "diver", "Patient"),
        ("The diver was frightened by the shark.", "diver", "Patient"),
        ("The diver was frightened by the shark.", "shark", "Agent"),
        ("The diver observed the shark.", "diver", "Agent"),
        ("The diver observed the shark.", "shark", "Patient"),
        ("The shark was observed by the diver.", "shark", "Patient"),
        ("The shark was observed by the diver.", "diver", "Agent"),
        # ---- king / knight -------------------------------------------
        ("The king summoned the knight.", "king", "Agent"),
        ("The king summoned the knight.", "knight", "Patient"),
        ("The knight was summoned by the king.", "knight", "Patient"),
        ("The knight was summoned by the king.", "king", "Agent"),
        ("The knight protected the king.", "knight", "Agent"),
        ("The knight protected the king.", "king", "Patient"),
        ("The king was protected by the knight.", "king", "Patient"),
        ("The king was protected by the knight.", "knight", "Agent"),
        # ---- journalist / editor ------------------------------------
        ("The journalist interviewed the editor.", "journalist", "Agent"),
        ("The journalist interviewed the editor.", "editor", "Patient"),
        ("The editor was interviewed by the journalist.", "editor", "Patient"),
        ("The editor was interviewed by the journalist.", "journalist", "Agent"),
        ("The editor reviewed the journalist.", "editor", "Agent"),
        ("The editor reviewed the journalist.", "journalist", "Patient"),
        ("The journalist was reviewed by the editor.", "journalist", "Patient"),
        ("The journalist was reviewed by the editor.", "editor", "Agent"),
    ]

    assert len(examples) == 152, f"Expected 152 examples, got {len(examples)}"
    return examples


def get_dataset_statistics(dataset: List[Tuple[str, str, str]]) -> None:
    """Print basic statistics about the dataset."""
    labels = [label for _, _, label in dataset]
    agents = labels.count("Agent")
    patients = labels.count("Patient")
    print(f"Total examples : {len(dataset)}")
    print(f"  Agent  labels: {agents}")
    print(f"  Patient labels: {patients}")

    # Count active vs passive sentences
    active_count = sum(1 for s, _, _ in dataset if " was " not in s and " were " not in s)
    passive_count = len(dataset) - active_count
    print(f"  Active  sentences: {active_count}")
    print(f"  Passive sentences: {passive_count}")

    # Check that all nouns appear as both Agent and Patient
    noun_roles: Dict[str, set] = {}
    for _, noun, label in dataset:
        noun_roles.setdefault(noun, set()).add(label)
    both_roles = {n for n, r in noun_roles.items() if len(r) == 2}
    print(f"  Nouns with both Agent & Patient labels: {len(both_roles)} / {len(noun_roles)}")

## 6. Feature Extraction Pipeline

In [7]:
def build_feature_matrix(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    dataset: List[Tuple[str, str, str]],
    layer: int = -1,
    device: str = "cpu",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract hidden-state features for every example in the dataset.

    Args:
        model: Language model with output_hidden_states=True.
        tokenizer: Corresponding tokenizer.
        dataset: List of (sentence, target_word, label) triples.
        layer: Transformer layer index to use for representations.
        device: Device string.

    Returns:
        X: np.ndarray of shape (n_examples, hidden_size)
        y: np.ndarray of shape (n_examples,) with string labels
    """
    X, y = [], []
    skipped = []

    for i, (sentence, target_word, label) in enumerate(dataset):
        try:
            vec = extract_hidden_state(
                model, tokenizer, sentence, target_word,
                layer=layer, device=device
            )
            X.append(vec)
            y.append(label)
        except ValueError as e:
            print(f"[WARNING] Skipping example {i}: {e}")
            skipped.append(i)

        if (i + 1) % 25 == 0:
            print(f"  Processed {i + 1}/{len(dataset)} examples...")

    if skipped:
        print(f"[WARNING] {len(skipped)} examples were skipped due to tokenization issues.")

    return np.array(X), np.array(y)

## 7. Classifier Training and Evaluation

In [8]:
def train_and_evaluate(
    X: np.ndarray,
    y: np.ndarray,
    n_train: int = 125,
    cv_folds: int = 5,
    random_state: int = 42,
    c_value: float = 3.0,
) -> LogisticRegression:
    """
    Split data, run cross-validation on the training set, and evaluate
    the final classifier on the held-out test set.

    Args:
        X: Feature matrix of shape (n_examples, hidden_size).
        y: Label array of shape (n_examples,).
        n_train: Number of training examples (remainder used for testing).
        cv_folds: Number of cross-validation folds.
        random_state: Random seed for reproducibility.
        c_value: Inverse regularization strength for LogisticRegression.

    Returns:
        Trained LogisticRegression classifier.
    """
    n_test = len(X) - n_train
    print(f"Dataset size: {len(X)} total ({n_train} train, {n_test} test)")

    # Encode labels to integers for compatibility
    le = LabelEncoder()
    y_enc = le.fit_transform(y)  # Agent=0, Patient=1 (alphabetical)
    print(f"Label encoding: {list(zip(le.classes_, range(len(le.classes_))))}")

    # Split into train / test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc,
        train_size=n_train,
        test_size=n_test,
        random_state=random_state,
        stratify=y_enc,
    )
    print(f"Train class distribution: { {le.classes_[i]: (y_train == i).sum() for i in range(len(le.classes_))} }")
    print(f"Test  class distribution: { {le.classes_[i]: (y_test  == i).sum() for i in range(len(le.classes_))} }")

    # Define the probe classifier
    clf = LogisticRegression(
        max_iter=1000,
        random_state=random_state,
        solver="lbfgs",
        C=c_value,
    )

    # ------------------------------------------------------------------ #
    # Cross-validation on the training set                               #
    # ------------------------------------------------------------------ #
    print(f"\nRunning {cv_folds}-fold cross-validation on training set...")
    cv_scores = cross_val_score(clf, X_train, y_train, cv=cv_folds, scoring="accuracy")
    print(f"Cross-validation accuracy (per fold): {np.round(cv_scores, 4)}")
    print(f"Cross-validation accuracy (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # ------------------------------------------------------------------ #
    # Train final classifier on full training set                        #
    # ------------------------------------------------------------------ #
    clf.fit(X_train, y_train)

    # ------------------------------------------------------------------ #
    # Evaluate on held-out test set                                      #
    # ------------------------------------------------------------------ #
    test_accuracy = clf.score(X_test, y_test)
    print(f"\nTest accuracy: {test_accuracy:.4f}")

    return clf

## 8. Main Experiment Pipeline

In [9]:
def run_probing_experiment(
    model_name: str = "CohereForAI/aya-expanse-8b",
    layer: int = -1,
    n_train: int = 125,
    cv_folds: int = 5,
    random_state: int = 42,
    c_value: float = 3.0,
) -> LogisticRegression:
    """
    End-to-end probing experiment for semantic role encoding.

    Steps:
    1. Load model and tokenizer.
    2. Build the Agent/Patient dataset.
    3. Extract hidden-state representations.
    4. Train a linear probe (logistic regression) with cross-validation.
    5. Report test accuracy.

    Args:
        model_name: HuggingFace model identifier.
        layer: Which transformer layer to probe (-1 = last).
        n_train: Number of training examples.
        cv_folds: Number of CV folds.
        random_state: Seed for reproducibility.
        c_value: Inverse regularization strength for LogisticRegression.

    Returns:
        Trained LogisticRegression probe classifier.
    """
    # Step 1: Load model
    print("=" * 60)
    print("STEP 1: Loading model and tokenizer")
    print("=" * 60)
    model, tokenizer, device = load_model_and_tokenizer(model_name)

    # Step 2: Build dataset
    print("\n" + "=" * 60)
    print("STEP 2: Building dataset")
    print("=" * 60)
    dataset = create_dataset()
    get_dataset_statistics(dataset)

    # Step 3: Extract hidden states
    print("\n" + "=" * 60)
    print(f"STEP 3: Extracting hidden states (layer={layer})")
    print("=" * 60)
    X, y = build_feature_matrix(model, tokenizer, dataset, layer=layer, device=device)
    print(f"Feature matrix shape: {X.shape}")

    # Cache artifacts for diagnostics so later cells do not need to reload model/tokenizer
    globals()["probing_artifacts"] = {
        "model": model,
        "tokenizer": tokenizer,
        "device": device,
        "dataset": dataset,
        "X": X,
        "y": y,
        "layer": layer,
        "n_train": n_train,
        "random_state": random_state,
        "c_value": c_value,
    }

    # Step 4 + 5: Train and evaluate
    print("\n" + "=" * 60)
    print("STEP 4: Training probe and evaluating")
    print("=" * 60)
    clf = train_and_evaluate(
        X,
        y,
        n_train=n_train,
        cv_folds=cv_folds,
        random_state=random_state,
        c_value=c_value,
    )

    # Keep classifier available for diagnostics
    globals()["trained_classifier"] = clf

    print("\n" + "=" * 60)
    print("Experiment complete.")
    print("=" * 60)
    return clf

## 9. Run the Experiment

> **Note:** Running the cell below requires a GPU and will download the ~16 GB `aya-expanse-8b` model weights. Estimated runtime: ~10–20 minutes on a Colab T4 GPU.

In [12]:
# Run the full probing experiment
trained_classifier = run_probing_experiment(
    model_name="CohereForAI/aya-expanse-8b",
    layer=25,       # Use the last transformer layer
    n_train=125,    # 125 training examples, 25 test examples
    cv_folds=5,     # 5-fold cross-validation
    random_state=42,
    c_value=3.0,
)

STEP 1: Loading model and tokenizer
Loading tokenizer for 'CohereForAI/aya-expanse-8b'...
Loading model on device: cuda


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded successfully.

STEP 2: Building dataset
Total examples : 152
  Agent  labels: 76
  Patient labels: 76
  Active  sentences: 76
  Passive sentences: 76
  Nouns with both Agent & Patient labels: 37 / 37

STEP 3: Extracting hidden states (layer=25)


KeyboardInterrupt: 

## 10. (Optional) Layer-wise Probing

Run the probe at every transformer layer to see which layer best encodes semantic roles.

In [ ]:
def extract_all_layers(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    dataset: List[Tuple[str, str, str]],
    device: str,
) -> Tuple[Dict[int, np.ndarray], np.ndarray]:
    """
    Run ONE forward pass per example and cache hidden states for ALL layers.

    Args:
        model: Language model with output_hidden_states=True.
        tokenizer: Corresponding tokenizer.
        dataset: List of (sentence, target_word, label) triples.
        device: Device string.

    Returns:
        layer_X: Dict mapping layer_index -> np.ndarray of shape (n_examples, hidden_size)
        y: np.ndarray of shape (n_examples,) with string labels
    """
    n_layers = model.config.num_hidden_layers
    # Pre-allocate lists for each layer (0 = embedding, 1..n = transformer layers)
    layer_vecs = {i: [] for i in range(n_layers + 1)}
    y = []
    skipped = []

    for i, (sentence, target_word, label) in enumerate(dataset):
        try:
            inputs = tokenizer(sentence, return_tensors="pt")
            input_ids = inputs["input_ids"][0]

            positions = find_token_positions(tokenizer, input_ids, target_word)
            if not positions:
                raise ValueError(
                    f"Target word '{target_word}' not found in tokenized sentence: '{sentence}'"
                )

            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)

            # Extract hidden states for ALL layers in this single forward pass
            for layer_idx in range(n_layers + 1):
                token_vecs = outputs.hidden_states[layer_idx][0, positions, :]
                vec = token_vecs.mean(dim=0).float().cpu().numpy()
                layer_vecs[layer_idx].append(vec)

            y.append(label)
        except ValueError as e:
            print(f"[WARNING] Skipping example {i}: {e}")
            skipped.append(i)

        if (i + 1) % 25 == 0:
            print(f"  Processed {i + 1}/{len(dataset)} examples...")

    if skipped:
        print(f"[WARNING] {len(skipped)} examples were skipped.")

    layer_X = {k: np.array(v) for k, v in layer_vecs.items()}
    return layer_X, np.array(y)


def run_layerwise_probing(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    dataset: List[Tuple[str, str, str]],
    device: str,
    n_train: int = 125,
    cv_folds: int = 5,
    random_state: int = 42,
    c_value: float = 3.0,
) -> Dict[int, Dict[str, float]]:
    """
    Probe each transformer layer and return mean CV accuracy and test accuracy per layer.

    Uses a single forward pass per example to extract all layers at once,
    instead of re-running the model for each layer.

    Args:
        model: Loaded language model.
        tokenizer: Corresponding tokenizer.
        dataset: List of (sentence, target_word, label) triples.
        device: Device string.
        n_train: Number of training examples for each layer probe.
        cv_folds: Number of CV folds.
        random_state: Seed.
        c_value: Inverse regularization strength for LogisticRegression.

    Returns:
        Dictionary mapping layer index -> {"cv_accuracy": float, "test_accuracy": float}.
    """
    n_layers = model.config.num_hidden_layers
    print(f"Model has {n_layers} transformer layers (+ embedding layer).")

    # Single pass: extract all layers at once
    print("\nExtracting hidden states for ALL layers in a single pass...")
    layer_X, y = extract_all_layers(model, tokenizer, dataset, device)
    print(f"Extraction complete. Feature shape per layer: {layer_X[0].shape}")

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    n_test = len(y_enc) - n_train

    results = {}
    for layer_idx in range(n_layers + 1):
        X = layer_X[layer_idx]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y_enc,
            train_size=n_train,
            test_size=n_test,
            random_state=random_state,
            stratify=y_enc,
        )

        clf = LogisticRegression(max_iter=1000, random_state=random_state, C=c_value)
        cv_scores = cross_val_score(clf, X_train, y_train, cv=cv_folds, scoring="accuracy")
        cv_mean = cv_scores.mean()

        # Also evaluate on the held-out test set
        clf.fit(X_train, y_train)
        test_acc = clf.score(X_test, y_test)

        results[layer_idx] = {"cv_accuracy": cv_mean, "test_accuracy": test_acc}
        print(f"  Layer {layer_idx:3d}: CV accuracy = {cv_mean:.4f}, test accuracy = {test_acc:.4f}")

    best_layer_cv = max(results, key=lambda k: results[k]["cv_accuracy"])
    best_layer_test = max(results, key=lambda k: results[k]["test_accuracy"])
    print(f"\nBest layer (CV):   {best_layer_cv} (accuracy = {results[best_layer_cv]['cv_accuracy']:.4f})")
    print(f"Best layer (test): {best_layer_test} (accuracy = {results[best_layer_test]['test_accuracy']:.4f})")
    return results, layer_X, y_enc


# Uncomment to run layer-wise probing
model, tokenizer, device = load_model_and_tokenizer()
dataset = create_dataset()
layer_results, layer_X, y_enc = run_layerwise_probing(
    model,
    tokenizer,
    dataset,
    device,
    c_value=3.0,
)

## 11. Diagnostic: Which test examples are classified correctly/incorrectly?

Check whether the same examples are misclassified across layers — a sign that the probe relies on a layer-independent feature (e.g., word identity) rather than contextual semantic role information.

In [11]:
# Short diagnostic: print misclassified test sentences for one layer
# Reuses cached artifacts from run_probing_experiment (no model/tokenizer reload).

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import numpy as np

if "probing_artifacts" not in globals():
    raise RuntimeError(
        "Run Cell 23 first (the main experiment cell) so features are cached in probing_artifacts."
    )

art = probing_artifacts
dataset = art["dataset"]
X = art["X"]
y = art["y"]
layer = art["layer"]
n_train = art["n_train"]
random_state = art["random_state"]
c_value = art["c_value"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

all_idx = np.arange(len(y_enc))
train_idx, test_idx = train_test_split(
    all_idx,
    train_size=n_train,
    test_size=len(y_enc) - n_train,
    random_state=random_state,
    stratify=y_enc,
)

# Use existing trained probe if available, else train quickly on cached features
if "trained_classifier" in globals():
    clf = trained_classifier
else:
    clf = LogisticRegression(max_iter=1000, random_state=random_state, C=c_value)
    clf.fit(X[train_idx], y_enc[train_idx])

y_test = y_enc[test_idx]
y_pred = clf.predict(X[test_idx])

wrong_mask = y_pred != y_test
wrong_idx = test_idx[wrong_mask]
wrong_pred = y_pred[wrong_mask]
wrong_true = y_test[wrong_mask]

print(f"Layer {layer} | Misclassified: {wrong_mask.sum()} / {len(test_idx)}\n")
for i, (idx, pred, true) in enumerate(zip(wrong_idx, wrong_pred, wrong_true), 1):
    sentence, target_word, _ = dataset[idx]
    true_label = le.inverse_transform([true])[0]
    pred_label = le.inverse_transform([pred])[0]
    print(f"[{i:02d}] idx={idx:3d} word='{target_word}' true={true_label} pred={pred_label} | {sentence}")

Layer 15 | Misclassified: 9 / 27

[01] idx= 50 word='nurse' true=Patient pred=Agent | The nurse was examined by the doctor.
[02] idx= 40 word='lion' true=Agent pred=Patient | The lion caught the zebra.
[03] idx=120 word='pilot' true=Agent pred=Patient | The pilot warned the passenger.
[04] idx= 28 word='fox' true=Agent pred=Patient | The fox chased the bird.
[05] idx=132 word='diver' true=Agent pred=Patient | The diver observed the shark.
[06] idx=118 word='eagle' true=Patient pred=Agent | The eagle was evaded by the rabbit.
[07] idx= 14 word='man' true=Patient pred=Agent | The man was helped by the woman.
[08] idx=  4 word='cat' true=Agent pred=Patient | The cat scratched the dog.
[09] idx=104 word='scientist' true=Agent pred=Patient | The scientist observed the assistant.


In [13]:
# Correctly classified test sentences
# Requires the previous diagnostic cell (Cell 27) to have been run first.

correct_mask = ~wrong_mask
correct_idx = test_idx[correct_mask]
correct_pred = y_pred[correct_mask]
correct_true = y_test[correct_mask]

print(f"Layer {layer} | Correctly classified: {correct_mask.sum()} / {len(test_idx)}\n")
for i, (idx, pred, true) in enumerate(zip(correct_idx, correct_pred, correct_true), 1):
    sentence, target_word, _ = dataset[idx]
    true_label = le.inverse_transform([true])[0]
    print(f"[{i:02d}] idx={idx:3d} word='{target_word}' label={true_label} | {sentence}")

Layer 15 | Correctly classified: 18 / 27

[01] idx= 43 word='lion' label=Agent | The zebra was caught by the lion.
[02] idx=  1 word='cat' label=Patient | The dog chased the cat.
[03] idx=129 word='diver' label=Patient | The shark frightened the diver.
[04] idx=149 word='journalist' label=Patient | The editor reviewed the journalist.
[05] idx=  3 word='dog' label=Agent | The cat was chased by the dog.
[06] idx= 57 word='mouse' label=Patient | The cat caught the mouse.
[07] idx=143 word='knight' label=Agent | The king was protected by the knight.
[08] idx=119 word='rabbit' label=Agent | The eagle was evaded by the rabbit.
[09] idx=105 word='assistant' label=Patient | The scientist observed the assistant.
[10] idx= 47 word='zebra' label=Agent | The lion was kicked by the zebra.
[11] idx= 29 word='bird' label=Patient | The fox chased the bird.
[12] idx= 46 word='lion' label=Patient | The lion was kicked by the zebra.
[13] idx=103 word='rider' label=Agent | The horse was guided by the ride